# Learning PIDs on Pendulum with LoPIDN

## 0. Setup

In [ ]:
import sys
sys.path.append('./src/lopidn')

In [ ]:
import gymnasium as gym
import random
import torch as th
from tqdm import tqdm
import yaml
import numpy as np

from config import RunConfig
from data import generate_dataset
from optimization import optimize_controller, optimize_LMN
import utils
from models import LocalModelNetwork, CriticLMN

In [ ]:
with open(f'./config/Pendulum.yaml', "r") as f:
    args = yaml.safe_load(f)
args = RunConfig(**args)
args.load_modules()

In [ ]:
random.seed(args.seed)
np.random.seed(args.seed)
th.manual_seed(args.seed)

## 1. Generate a state-action dataset

In [ ]:
# Create environment
env = gym.make(args.env_id)

In [ ]:
# Create dataset
dataset = generate_dataset(env_id=args.env_id,
                           agent=f'./{args.rl_agent}',
                           save_path=f'./trained_rl/datasets/{args.algorithm}/{args.env_id}.pkl',
                           algorithm=args.algorithm_class,
                           n_episodes=args.data_n_episodes)

In [ ]:
# Initialize router
router = CriticLMN(n_inputs=env.observation_space.shape[0],
                   n_outputs=env.action_space.shape[0],
                   n_controllers=args.n_controllers,
                   controller_class=args.controller_class,
                   controller_target=args.target,
                   controller_bound = (env.action_space.low, env.action_space.high),
                   critic=dataset.critic)

## 2. Optimize PIDs

In [ ]:
# Call optimization function
optimize_LMN(model=router,
             data=dataset,
             args=args)

## 4. Validate found controllers

In [ ]:
# Run several validation runs in environment
val_ret = utils.validate(controller=router,
                         env_id=args.env_id,
                         render_env=False,
                         n_episodes=args.validation_n_episodes)
print(f'Mean return: {np.mean(val_ret)} ± {np.std(val_ret)}')